# Zero‑Shot Next‑Word Prediction with LSTM‑Attention + Prompt Engineering

This notebook demonstrates how to **load a pre‑trained LSTM‑Attention language model**, craft a few simple prompts, and obtain the **first predicted word** without any fine‑tuning.  The whole pipeline runs in a few seconds on a free Google‑Colab GPU.

## What you will learn
- How additive (Bahdanau) attention works in an encoder‑decoder RNN.
- How to treat a prompt as a “question” that guides the model (prompt engineering).
- How to extract the attention weights for the first decoding step and visualise them.


In [ ]:
# Install required packages (run once)
!pip install -q torch==2.2.0 tqdm


## 1️⃣ Download the checkpoint and vocabulary
We use a small checkpoint (≈ 5 MB) trained on the WikiText‑2 dataset.  If you already have a checkpoint, replace the URL below.

In [ ]:
import os, json, urllib.request, pathlib\nfrom pathlib import Path\n\n# Directory to store assets\nasset_dir = Path('assets')\nasset_dir.mkdir(exist_ok=True)\n\n# URLs – replace with your own if you host the files elsewhere\nckpt_url = 'https://github.com/your-org/lstm-attn-wikitext/releases/download/v0.1/lstm_attn_wikitext.pt'\nvocab_url = 'https://github.com/your-org/lstm-attn-wikitext/releases/download/v0.1/vocab.json'\n\nckpt_path = asset_dir / 'lstm_attn_wikitext.pt'\nvocab_path = asset_dir / 'vocab.json'\n\ndef download(url, path):\n    if not path.exists():\n        print(f'Downloading {url} ...')\n        urllib.request.urlretrieve(url, path)\n        print('Done.')\n    else:\n        print(f'{path.name} already present.')\n\ndownload(ckpt_url, ckpt_path)\ndownload(vocab_url, vocab_path)\n

## 2️⃣ Load vocabulary and the checkpoint

In [ ]:
import torch\n\n# Load vocab (word → index)\nwith open(vocab_path, 'r') as f:\n    vocab = json.load(f)\nidx2word = {i:w for w,i in vocab.items()}\n\nsos_idx = vocab['<sos>']\neos_idx = vocab['<eos>']\npad_idx = vocab['<pad>']\n\n# Load checkpoint (state dict)\nstate = torch.load(ckpt_path, map_location='cpu')\n

## 3️⃣ Model definition (LSTM encoder + additive attention decoder)

In [ ]:
class LSTMAttnLM(torch.nn.Module):\n    def __init__(self, vocab_sz, emb_dim=256, hidden=256):\n        super().__init__()\n        self.emb = torch.nn.Embedding(vocab_sz, emb_dim, padding_idx=pad_idx)\n        self.encoder = torch.nn.LSTM(emb_dim, hidden, batch_first=True)\n        # additive attention parameters\n        self.W1 = torch.nn.Linear(hidden, hidden, bias=False)\n        self.W2 = torch.nn.Linear(hidden, hidden, bias=False)\n        self.v  = torch.nn.Linear(hidden, 1, bias=False)\n        self.decoder = torch.nn.LSTMCell(emb_dim + hidden, hidden)\n        self.out = torch.nn.Linear(hidden, vocab_sz)\n\n    def forward(self, src_ids):\n        # Encoder\n        src_emb = self.emb(src_ids)\n        enc_out, (h_n, _) = self.encoder(src_emb)  # enc_out: (B,T,H)\n        # Decoder – single step (first token)\n        dec_h = h_n.squeeze(0)                     # (B,H)\n        dec_c = torch.zeros_like(dec_h)\n        # attention scores\n        score = self.v(torch.tanh(self.W1(enc_out) + self.W2(dec_h).unsqueeze(1)))  # (B,T,1)\n        attn = torch.softmax(score.squeeze(-1), dim=1)                     # (B,T)\n        context = torch.sum(attn.unsqueeze(-1) * enc_out, dim=1)            # (B,H)\n        # first input token = <sos>\n        sos_tensor = torch.full((src_ids.size(0),), sos_idx, dtype=torch.long, device=src_ids.device)\n        sos_emb = self.emb(sos_tensor)\n        dec_input = torch.cat([sos_emb, context], dim=1)\n        dec_h, dec_c = self.decoder(dec_input, (dec_h, dec_c))\n        logits = self.out(dec_h)\n        return logits, attn\n\n# Instantiate and load weights\nmodel = LSTMAttnLM(vocab_sz=len(vocab))\nmodel.load_state_dict(state['model'])\nmodel.eval()\n

## 4️⃣ Prompt engineering – simple templates\nWe treat a prompt as a short sentence that the model will *read* before we ask it to predict the next word.

In [ ]:
templates = [\n    'Complete the sentence: {}',\n    'What comes next? {}',\n    'Continue: {}'\n]\n\ndef build_prompt(text, tmpl_id=0):\n    prompt = templates[tmpl_id].format(text)\n    tokens = prompt.lower().split()\n    ids = [vocab.get(tok, vocab['<unk>']) for tok in tokens]\n    return torch.tensor([ids], dtype=torch.long)\n

## 5️⃣ Inference – get the first predicted word and attention map

In [ ]:
def predict_next_word(prompt_ids):\n    with torch.no_grad():\n        logits, attn = model(prompt_ids)\n        prob = torch.softmax(logits, dim=-1)\n        top_idx = prob.argmax(dim=-1).item()\n        return idx2word[top_idx], attn.squeeze(0).cpu().numpy()\n\n# Example usage\nuser_text = 'the quick brown fox'\nprompt_ids = build_prompt(user_text, tmpl_id=0)\nnext_word, attn_weights = predict_next_word(prompt_ids)\nprint(f'Prompt : {templates[0].format(user_text)}')\nprint(f'Predicted next word → {next_word}')

## 6️⃣ Visualise the attention distribution over the prompt

In [ ]:
import matplotlib.pyplot as plt\n\ndef show_attention(prompt, attn_weights):\n    words = prompt.lower().split()\n    plt.figure(figsize=(6,1))\n    plt.imshow(attn_weights[:len(words)][None, :], cmap='viridis')\n    plt.xticks(range(len(words)), words, rotation=45, ha='right')\n    plt.yticks([])\n    plt.title('Attention over prompt (first decoding step)')\n    plt.show()\n\nshow_attention(templates[0].format(user_text), attn_weights)

---\n### 🎉 What you have now\n- A **pre‑trained LSTM‑Attention** language model loaded from a checkpoint.\n- A few **prompt templates** that steer the model’s prediction.\n- Code that **predicts the very first word** after the prompt (zero‑shot).\n- A **visualisation** of the attention weights, showing which part of the prompt the model focused on.\n\nYou can now experiment by changing the prompt, trying different templates, or extending the decoder loop to generate full sentences.